# Nimzo — scaled data + fixed pipeline retrain (base_v4)

What changed since base_v3:
1. **MCTS bugs fixed**: fifty-move/threefold terminal detection, a transposition
   table that actually merges duplicate positions within a search, and real
   cross-move tree reuse (previously every move silently restarted the search
   tree from scratch). Measured on the existing base_v3 checkpoint: score vs.
   Stockfish at 1-10 nodes roughly doubled (2.5% → 5%), confirming these were
   real but partial contributors to the weak Elo -- most of the gap is still
   data/capacity, which this run targets.
2. **Data-ratio bug fixed**: training previously concatenated puzzle + eval
   shards and shuffled uniformly over raw record count, so puzzles went from
   100% of every batch to ~15% as eval data was added (this is why policy
   accuracy collapsed 59%→39% when eval data was first added). `train.py` now
   takes puzzle and eval data as two separate files with a `RatioSampler`
   holding a fixed target ratio (40% puzzle / 60% eval by default) regardless of
   how large either file is.
3. **Cosine LR schedule with warmup** added (previously flat LR the whole run).
4. **4x more puzzle data**: 4,000,000 positions (up from 1,000,000), packed
   locally and included directly in the uploaded package -- no need to re-pull.
5. **3x more eval data this run**: pulling 15,000,000 Stockfish-evaluated
   positions (up from 5,000,000), streamed the same proven way (no `zstdcat`
   dependency, decompressed in Python via the `zstandard` package).

Training fresh (not resumed from base_v3) since the LR schedule needs a clean
step-0 start to shape correctly over the new, much longer step budget.

Before running:
1. Turn on GPU: **Settings (right sidebar) → Accelerator → GPU T4 x2**.
2. Add the dataset: **Add Input → Upload → `nimzo_kaggle_package_v4.zip`**.
3. **Save & Run All (commit)** — this is a long run (~150K steps, expect a
   couple of hours), runs in the background so you can close the tab.
4. Download `runs_base_v4.zip` from the notebook's Output tab when done.

In [ ]:
!pip install -q python-chess pyyaml tqdm zstandard
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — did you enable the GPU accelerator?')

In [ ]:
import zipfile, glob, os, shutil

extracted_pkg = glob.glob('/kaggle/input/**/kaggle_package_v4', recursive=True)
src_zip = glob.glob('/kaggle/input/**/nimzo_kaggle_package_v4.zip', recursive=True) \
    or glob.glob('/kaggle/input/**/*.zip', recursive=True)

dest = '/kaggle/working/extracted/kaggle_package_v4'
if os.path.exists(dest):
    shutil.rmtree(dest)

if extracted_pkg:
    shutil.copytree(extracted_pkg[0], dest)
elif src_zip:
    with zipfile.ZipFile(src_zip[0]) as z:
        z.extractall('/kaggle/working/extracted')
else:
    all_input_files = glob.glob('/kaggle/input/**/*', recursive=True)
    raise FileNotFoundError(
        'Could not find kaggle_package_v4/ or nimzo_kaggle_package_v4.zip under /kaggle/input/. '
        'All files found: ' + str(all_input_files)
    )

os.chdir(dest)
print(os.listdir('.'))
print(os.listdir('data'))

In [ ]:
import sys
sys.path.insert(0, dest)
os.environ['PYTHONPATH'] = dest

# Stream 15M eval positions -- same proven approach as base_v2/v3, decompressed
# in Python (no zstdcat dependency, which isn't on the Kaggle image).
!curl -sL https://database.lichess.org/lichess_db_eval.jsonl.zst | \
  python -m nimzo.data.pack_evals --input - --out data --limit 15000000

In [ ]:
evals_size = os.path.getsize('data/evals_train.bin')
print(f'evals_train.bin size: {evals_size:,} bytes ({evals_size // 40:,} positions)')
assert evals_size > 100_000_000, (
    f'evals_train.bin is only {evals_size} bytes -- the eval data pull failed or '
    'produced far less than expected. Check the curl/pack_evals cell output above '
    'before continuing.'
)
!ls -la data/

In [ ]:
from nimzo.train import train

# Fresh run (clean LR schedule from step 0). Puzzle:eval ratio held at 40:60
# regardless of the 4M vs 15M file-size imbalance (RatioSampler, not
# a plain concatenation -- see nimzo/train.py and nimzo/data/shards.py).
train(
    config_path='configs/base.yaml',
    data_path='data/puzzles_train.bin',
    eval_data_path='data/evals_train.bin',
    puzzle_ratio=0.4,
    run_name='base_v4',
    resume=None,
    device='cuda',
    steps=150000,
)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/runs_base_v4', 'zip', os.path.join(dest, 'runs/base_v4'))
print('Saved /kaggle/working/runs_base_v4.zip')
print(os.listdir(os.path.join(dest, 'runs/base_v4')))